# Node-level Keyword Extraction (Reviews)

This notebook experiments with extracting **top-N representative keywords or short phrases** for each node, split by **positive vs negative** sentiment.

Goals:
- Filter to clearly positive/negative reviews (configurable thresholds).
- Language-aware preprocessing:
  - English: tokenization, lowercasing, stopword removal.
  - Korean: morphological analysis with noun/adjective extraction (falls back to simple tokenization if a tagger is unavailable).
- Candidate keywords via:
  - frequency counts
  - optional TF-IDF (node-level)
  - optional bi-grams
- Output ranked keyword lists per node and polarity.

Notes:
- This is designed for clarity/experimentation, not production optimization.
- Use the outputs later to pick representative reviews.


In [7]:
# Core imports
import os
import re
import sqlite3
from pathlib import Path
from collections import Counter

import pandas as pd


In [8]:
# ---- Config ----
#DB_PATH = Path("../data/app.sqlite")
DB_PATH = Path("../data/app_simulated.sqlite")

MECAB_PATH = "/usr/lib/x86_64-linux-gnu/mecab/dic/mecab-ko-dic"

# Sentiment filtering
SENTIMENT_SCALE = "auto"  # "auto", "zero_to_one", or "minus_one_to_one"
POS_THRESHOLD_0_1 = 0.6
NEG_THRESHOLD_0_1 = 0.4
POS_THRESHOLD_M11 = 0.2
NEG_THRESHOLD_M11 = -0.2
MIN_CONFIDENCE = 0.0  # set >0 to require confidence (e.g., 0.6)

# Keyword extraction
TOP_N = 3
USE_TFIDF = True
MIN_TOKEN_LEN = 2  # applies to English; Korean uses a separate filter

# Keyword quality toggles
USE_SURFACE_NORMALIZATION = True
USE_NARRATIVE_STOPWORDS = True
EXCLUDE_NODE_NAME = True
USE_NGRAMS = False

# If your sentiments are stored by multiple methods/versions, select them here.
SENTIMENT_METHOD = None  # e.g., "vader", or None to keep all
SENTIMENT_VERSION = None  # e.g., "1.0", or None to keep all


In [9]:
# ---- Load reviews + sentiment ----
if not os.path.exists(DB_PATH):
    raise FileNotFoundError(f"DB not found: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

query = '''
SELECT
  r.id AS review_id,
  r.entity_id,
  n.name AS node_name,
  r.content AS text,
  r.entity_name,
  r.created_at,
  s.sentiment_raw,
  s.confidence,
  s.method,
  s.version
FROM review r
LEFT JOIN nodes n ON n.id = r.entity_id
JOIN review_sentiment s ON s.review_id = r.id
WHERE r.entity_id IS NOT NULL
'''

params = {}
if SENTIMENT_METHOD is not None:
    query += " AND s.method = :method"
    params["method"] = SENTIMENT_METHOD
if SENTIMENT_VERSION is not None:
    query += " AND s.version = :version"
    params["version"] = SENTIMENT_VERSION

reviews = pd.read_sql_query(query, conn, params=params)
conn.close()

reviews.head()


,review_id,entity_id,node_name,text,entity_name,created_at,sentiment_raw,confidence,method,version
0,1,39,교촌간장,교촌간장은 기대보다 실망이었다,교촌간장,2026-02-05T02:50:44.296095+00:00,-1.0,0.5,simulated,v1
1,2,39,교촌간장,교촌간장은 너무 별로였다,교촌간장,2026-02-05T02:50:44.296095+00:00,-2.0,1.0,simulated,v1
2,3,40,교촌레드,교촌레드은 자주 사먹고 싶다,교촌레드,2026-02-05T02:50:44.296095+00:00,2.0,1.0,simulated,v1
3,4,109,한돈 그릴 비엔나,한돈 그릴 비엔나은 꽤 만족스러웠다,한돈 그릴 비엔나,2026-02-05T02:50:44.296095+00:00,1.0,0.5,simulated,v1
4,5,39,교촌간장,교촌간장은 정말 마음에 들었다,교촌간장,2026-02-05T02:50:44.296095+00:00,2.0,1.0,simulated,v1


## 1) Filter to clearly positive/negative reviews

We detect the sentiment scale and apply conservative thresholds:
- For 0..1 scores: positive ≥ **0.6**, negative ≤ **0.4**
- For -1..1 scores: positive ≥ **0.2**, negative ≤ **-0.2**

Adjust the thresholds above if needed. Optionally apply a minimum confidence.


In [10]:
def detect_scale(series, mode="auto"):
    if mode == "zero_to_one":
        return "zero_to_one"
    if mode == "minus_one_to_one":
        return "minus_one_to_one"

    # auto
    min_v = series.min()
    max_v = series.max()
    if min_v >= -1.1 and max_v <= 1.1 and (min_v < 0):
        return "minus_one_to_one"
    return "zero_to_one"


def label_sentiment(score, scale):
    if scale == "minus_one_to_one":
        if score >= POS_THRESHOLD_M11:
            return "positive"
        if score <= NEG_THRESHOLD_M11:
            return "negative"
        return None
    # zero_to_one
    if score >= POS_THRESHOLD_0_1:
        return "positive"
    if score <= NEG_THRESHOLD_0_1:
        return "negative"
    return None

scale = detect_scale(reviews["sentiment_raw"], SENTIMENT_SCALE)

filtered = reviews.copy()
if MIN_CONFIDENCE > 0:
    filtered = filtered[filtered["confidence"].fillna(0) >= MIN_CONFIDENCE]

filtered["polarity"] = filtered["sentiment_raw"].apply(lambda s: label_sentiment(s, scale))
filtered = filtered[filtered["polarity"].notna()].reset_index(drop=True)

print("scale:", scale)
print("reviews kept:", len(filtered), "of", len(reviews))
filtered["polarity"].value_counts()


scale: zero_to_one
reviews kept: 8561 of 8561


polarity
negative    5093
positive    3468
Name: count, dtype: int64

## 2) Language-aware preprocessing

English:
- tokenize by letters/numbers
- lowercase
- remove stopwords

Korean:
- prefer morphological analysis (Okt) to extract nouns/adjectives
- fallback to simple Hangul tokenization if tagger is unavailable


In [11]:
# --- English preprocessing ---

import nltk
from nltk.corpus import stopwords
nltk.download("stopwords", quiet=True)
EN_STOPWORDS = set(stopwords.words("english"))


# Customizable stopwords
EN_STOPWORDS_CUSTOM = {
    "ok","okay","yeah","yep","nope","hi","hello","thanks","thank","pls","please",
    "maybe","just","like","really","very","thing","stuff"
}
EN_STOPWORDS = EN_STOPWORDS.union(EN_STOPWORDS_CUSTOM)

EN_TOKEN_RE = re.compile(r"[a-zA-Z0-9']+")


def tokenize_english(text):
    tokens = [t.lower() for t in EN_TOKEN_RE.findall(text or "")]
    tokens = [t for t in tokens if t not in EN_STOPWORDS and len(t) >= MIN_TOKEN_LEN]
    return tokens


# --- Korean preprocessing ---
# POS allowlist: explicit + configurable
KOREAN_POS_ALLOW = {"NNG", "NNP", "VA"}

# Customizable stopwords (add as needed)
KOREAN_STOPWORDS = {
    "근데","그런데","같다","이건","저건","원래","무슨","진짜","정말",
    "그냥","좀","조금","항상","대박","완전","너무","약간","그리고",
    "하지만","그래서","왜","뭔가","그거","이거","저거","여기","거기",
    "저기","이렇게","그렇게","저렇게","있다","없다","하다","되다"
}

# Narrative / weak-semantic stopwords (kept separate)
KOREAN_NARRATIVE_STOPWORDS = {
    "근데","그냥","뭔가","원래"
}

# Simple canonicalization map (extend as needed)
KOREAN_CANON = {
    "맛나다": "맛있다",
    "맛있음": "맛있다",
    "맛있어요": "맛있다",
}

# Lightweight surface-form normalization
KOREAN_SURFACE_NORM = {
    "맛있": "맛있다",
    "비싸": "비싸다",
}

from konlpy.tag import Mecab
mecab = Mecab(dicpath=MECAB_PATH)
MECAB_CALLS = 0


def mecab_pos(text):
    global MECAB_CALLS
    MECAB_CALLS += 1
    return mecab.pos(text or "")


def tokenize_korean_with_pos(text):
    tokens = []
    tokens_with_pos = []
    for word, pos in mecab_pos(text):
        if pos not in KOREAN_POS_ALLOW:
            continue
        if len(word) < 2:
            continue
        # Normalize to canonical forms where applicable
        canon = KOREAN_CANON.get(word, word)
        if canon in KOREAN_STOPWORDS:
            continue
        tokens.append(canon)
        tokens_with_pos.append((canon, pos))
    return tokens, tokens_with_pos

KOREAN_TOKENIZER = "mecab"


def detect_language(text):
    # simple heuristic: Hangul => Korean
    return "ko" if re.search(r"[가-힣]", text or "") else "en"


def tokenize_with_pos(text):
    lang = detect_language(text)
    if lang == "ko":
        return tokenize_korean_with_pos(text)
    tokens = tokenize_english(text)
    return tokens, [(t, "EN") for t in tokens]

print("korean tokenizer:", KOREAN_TOKENIZER)


korean tokenizer: mecab


In [12]:
# ---- Explicit tokenization step (single source of truth) ----
filtered[["tokens", "tokens_with_pos"]] = filtered["text"].apply(
    lambda t: pd.Series(tokenize_with_pos(t))
)
filtered["tokens_joined"] = filtered["tokens"].apply(lambda t: " ".join(t))

ko_count = filtered["text"].str.contains(r"[가-힣]", na=False).sum()
print(f"Mecab calls: {MECAB_CALLS} (ko reviews: {ko_count})")

filtered[["text", "tokens"]].head()


Mecab calls: 8561 (ko reviews: 8561)


,text,tokens
0,교촌간장은 기대보다 실망이었다,"[교촌, 간장, 기대, 실망]"
1,교촌간장은 너무 별로였다,"[교촌, 간장]"
2,교촌레드은 자주 사먹고 싶다,"[교촌, 레드]"
3,한돈 그릴 비엔나은 꽤 만족스러웠다,"[그릴, 비엔나, 만족]"
4,교촌간장은 정말 마음에 들었다,"[교촌, 간장, 마음]"


## 3) Candidate keyword extraction

We create per-review tokens, then aggregate per node and polarity.
Optional bigrams are appended when `USE_BIGRAMS=True`.


In [13]:
def add_bigrams(tokens):
    return [f"{a} {b}" for a, b in zip(tokens, tokens[1:])]


def build_korean_bigrams_from_pos(tokens_with_pos):
    pairs = []
    for (w1, p1), (w2, p2) in zip(tokens_with_pos, tokens_with_pos[1:]):
        if p1 in {"NNG", "NNP"} and p2 in {"NNG", "NNP"}:
            pairs.append(f"{w1} {w2}")
        elif p1 == "VA" and p2 in {"NNG", "NNP"}:
            pairs.append(f"{w1} {w2}")
    return pairs


def sentiment_weight(confidence):
    # Weight keyword contributions by sentiment confidence.
    # Ambiguous reviews get lower weight.
    if confidence is None:
        return 1.0
    try:
        c = float(confidence)
    except Exception:
        return 1.0
    # squashing: 0.0 -> 0.5, 1.0 -> 1.5
    return 0.5 + c


def preprocess_reviews(df, use_norm=USE_SURFACE_NORMALIZATION, use_narrative=USE_NARRATIVE_STOPWORDS, use_ngrams=USE_NGRAMS, exclude_node=EXCLUDE_NODE_NAME):
    all_tokens = []
    all_weights = []

    for text, toks, toks_pos, conf, node_name in zip(
        df["text"].fillna(""),
        df["tokens"],
        df["tokens_with_pos"],
        df["confidence"].fillna(0),
        df["node_name"].fillna(""),
    ):
        lang = detect_language(text)

        if lang == "ko":
            if use_norm:
                toks = [KOREAN_SURFACE_NORM.get(t, t) for t in toks]
                toks_pos = [(KOREAN_SURFACE_NORM.get(t, t), p) for t, p in toks_pos]
            if use_narrative:
                keep = [t for t in toks if t not in KOREAN_NARRATIVE_STOPWORDS]
                keep_set = set(keep)
                toks = keep
                toks_pos = [(t, p) for t, p in toks_pos if t in keep_set]

        if exclude_node and node_name:
            toks = [t for t in toks if t != node_name]
            toks_pos = [(t, p) for t, p in toks_pos if t != node_name]

        if use_ngrams and lang == "ko":
            toks = toks + build_korean_bigrams_from_pos(toks_pos)

        all_tokens.append(toks)
        all_weights.append(sentiment_weight(conf))

    out = df.copy()
    out["tokens"] = all_tokens
    out["tokens_joined"] = [" ".join(toks) for toks in all_tokens]
    out["weight"] = all_weights
    return out

processed = preprocess_reviews(filtered)
processed.head()


,review_id,entity_id,node_name,text,entity_name,created_at,sentiment_raw,confidence,method,version,polarity,tokens,tokens_with_pos,tokens_joined,weight
0,1,39,교촌간장,교촌간장은 기대보다 실망이었다,교촌간장,2026-02-05T02:50:44.296095+00:00,-1.0,0.5,simulated,v1,negative,"[교촌, 간장, 기대, 실망]","[(교촌, NNG), (간장, NNG), (기대, NNG), (실망, NNG)]",교촌 간장 기대 실망,1.0
1,2,39,교촌간장,교촌간장은 너무 별로였다,교촌간장,2026-02-05T02:50:44.296095+00:00,-2.0,1.0,simulated,v1,negative,"[교촌, 간장]","[(교촌, NNG), (간장, NNG)]",교촌 간장,1.5
2,3,40,교촌레드,교촌레드은 자주 사먹고 싶다,교촌레드,2026-02-05T02:50:44.296095+00:00,2.0,1.0,simulated,v1,positive,"[교촌, 레드]","[(교촌, NNP), (레드, NNP)]",교촌 레드,1.5
3,4,109,한돈 그릴 비엔나,한돈 그릴 비엔나은 꽤 만족스러웠다,한돈 그릴 비엔나,2026-02-05T02:50:44.296095+00:00,1.0,0.5,simulated,v1,positive,"[그릴, 비엔나, 만족]","[(그릴, NNG), (비엔나, NNP), (만족, NNG)]",그릴 비엔나 만족,1.0
4,5,39,교촌간장,교촌간장은 정말 마음에 들었다,교촌간장,2026-02-05T02:50:44.296095+00:00,2.0,1.0,simulated,v1,positive,"[교촌, 간장, 마음]","[(교촌, NNG), (간장, NNG), (마음, NNG)]",교촌 간장 마음,1.5


In [15]:
# ---- Frequency-based keywords (weighted) ----

def keyword_counts(df, top_n=TOP_N):
    results = []
    for (entity_id, node_name, polarity), group in df.groupby(["entity_id", "node_name", "polarity"], dropna=False):
        counter = Counter()
        source_counter = Counter()
        for toks, w in zip(group["tokens"], group["weight"]):
            if not toks:
                continue
            # count once per review for source count
            source_counter.update(set(toks))
            # weighted frequency
            counter.update({t: w for t in toks})
        for token, score in counter.most_common(top_n):
            results.append({
                "entity_id": entity_id,
                "node_name": node_name,
                "polarity": polarity,
                "keyword": token,
                "score": float(score),
                "source_count": int(source_counter.get(token, 0)),
                "source": "count",
            })
    return pd.DataFrame(results)

count_df = keyword_counts(processed)
count_df.head()


,entity_id,node_name,polarity,keyword,score,source_count,source
0,1,쏘마치 (BHC),negative,마치,81.0,94,count
1,1,쏘마치 (BHC),negative,기대,26.0,26,count
2,1,쏘마치 (BHC),negative,실망,26.0,26,count
3,1,쏘마치 (BHC),positive,마치,78.0,63,count
4,1,쏘마치 (BHC),positive,만족,33.0,33,count


In [16]:
# ---- Comparison: base vs enhanced keywords (count) ----
base_processed = preprocess_reviews(
    filtered,
    use_norm=False,
    use_narrative=False,
    use_ngrams=False,
    exclude_node=False,
)
base_counts = keyword_counts(base_processed)

enhanced_processed = preprocess_reviews(
    filtered,
    use_norm=USE_SURFACE_NORMALIZATION,
    use_narrative=USE_NARRATIVE_STOPWORDS,
    use_ngrams=USE_NGRAMS,
    exclude_node=EXCLUDE_NODE_NAME,
)
enhanced_counts = keyword_counts(enhanced_processed)

sample_node = enhanced_counts["entity_id"].dropna().iloc[0] if not enhanced_counts.empty else None
if sample_node is not None:
    print("Sample node:", sample_node)
    print("\n[Base]")
    display(base_counts[base_counts["entity_id"] == sample_node]
            .sort_values("score", ascending=False).head(TOP_N)
            [["keyword", "score"]].reset_index(drop=True))
    print("\n[Enhanced]")
    display(enhanced_counts[enhanced_counts["entity_id"] == sample_node]
            .sort_values("score", ascending=False).head(TOP_N)
            [["keyword", "score"]].reset_index(drop=True))


Sample node: 1

[Base]


,keyword,score
0,마치,81.0
1,마치,78.0
2,만족,33.0



[Enhanced]


,keyword,score
0,마치,81.0
1,마치,78.0
2,만족,33.0


In [17]:
# ---- Optional TF-IDF per node (per polarity) ----
if USE_TFIDF:
    from sklearn.feature_extraction.text import TfidfVectorizer

    def keyword_tfidf(df, top_n=TOP_N):
        results = []
        for polarity, pgroup in df.groupby("polarity"):
            # Each node is treated as a single document
            docs = []
            node_keys = []
            node_token_sources = []

            for (entity_id, node_name), ngroup in pgroup.groupby(["entity_id", "node_name"], dropna=False):
                node_keys.append((entity_id, node_name))
                docs.append(" ".join(ngroup["tokens_joined"].tolist()))

                # source count per token (reviews containing the token)
                sc = Counter()
                for toks in ngroup["tokens"]:
                    sc.update(set(toks))
                node_token_sources.append(sc)

            if not docs:
                continue

            vectorizer = TfidfVectorizer(token_pattern=r"[^ ]+", lowercase=False)
            tfidf = vectorizer.fit_transform(docs)
            vocab = vectorizer.get_feature_names_out()

            for i, (entity_id, node_name) in enumerate(node_keys):
                row = tfidf.getrow(i)
                if row.nnz == 0:
                    continue
                top_idx = row.toarray().ravel().argsort()[-top_n:][::-1]
                for idx in top_idx:
                    score = row[0, idx]
                    if score <= 0:
                        continue
                    token = vocab[idx]
                    results.append({
                        "entity_id": entity_id,
                        "node_name": node_name,
                        "polarity": polarity,
                        "keyword": token,
                        "score": float(score),
                        "source_count": int(node_token_sources[i].get(token, 0)),
                        "source": "tfidf",
                    })

        return pd.DataFrame(results)

    tfidf_df = keyword_tfidf(processed)
else:
    tfidf_df = pd.DataFrame()

(tfidf_df.head() if not tfidf_df.empty else tfidf_df)


,entity_id,node_name,polarity,keyword,score,source_count,source
0,1,쏘마치 (BHC),negative,마치,0.991924,94,tfidf
1,1,쏘마치 (BHC),negative,실망,0.076961,26,tfidf
2,1,쏘마치 (BHC),negative,기대,0.076961,26,tfidf
3,2,K마라치킨 (처갓집 양념치킨),negative,처갓집,0.575356,118,tfidf
4,2,K마라치킨 (처갓집 양념치킨),negative,마라치,0.575356,118,tfidf


## 4) Combine results and display

We show a readable table per node and polarity. You can choose which source to display (`count` or `tfidf`).


In [18]:
def combine_results(count_df, tfidf_df):
    frames = [count_df]
    if not tfidf_df.empty:
        frames.append(tfidf_df)
    return pd.concat(frames, ignore_index=True)

results = combine_results(count_df, tfidf_df)
results.head()


,entity_id,node_name,polarity,keyword,score,source_count,source
0,1,쏘마치 (BHC),negative,마치,81.0,94,count
1,1,쏘마치 (BHC),negative,기대,26.0,26,count
2,1,쏘마치 (BHC),negative,실망,26.0,26,count
3,1,쏘마치 (BHC),positive,마치,78.0,63,count
4,1,쏘마치 (BHC),positive,만족,33.0,33,count


In [19]:
# ---- Persist keywords: table ----
conn = sqlite3.connect(DB_PATH)

In [20]:
conn.executescript(
        """ DELETE FROM node_review_keywords; """
)
conn.commit()

In [21]:
conn.execute("""
CREATE TABLE IF NOT EXISTS node_review_keywords (
  node_id INTEGER NOT NULL,
  polarity TEXT NOT NULL,
  keyword TEXT NOT NULL,
  score REAL NOT NULL,
  rank INTEGER NOT NULL,
  version TEXT NOT NULL,
  updated_at TEXT NOT NULL,
  FOREIGN KEY (node_id) REFERENCES nodes(id)
);
""")
conn.commit()

In [23]:
# ---- Persist keywords: insert top-N per node/polarity ----
VERSION = "kw_v1"
SOURCE = "count"

subset = results[results["source"] == SOURCE].copy()
subset["rank"] = (
    subset.groupby(["entity_id", "polarity"])["score"]
    .rank(method="first", ascending=False)
    .astype(int)
)
subset = subset[subset["rank"] <= TOP_N]
subset["version"] = VERSION
subset["updated_at"] = pd.Timestamp.utcnow().isoformat()

conn.execute("DELETE FROM node_review_keywords WHERE version = ?", (VERSION,))

rows = list(
    subset[
        ["entity_id", "polarity", "keyword", "score", "rank", "version", "updated_at"]
    ].itertuples(index=False, name=None)
)

conn.executemany(
    """
    INSERT INTO node_review_keywords
      (node_id, polarity, keyword, score, rank, version, updated_at)
    VALUES (?, ?, ?, ?, ?, ?, ?)
    """,
    rows,
)
conn.commit()

print("inserted rows:", len(rows))


inserted rows: 150


In [24]:
# ---- Verify join with node_review_stats ----
query = """
SELECT
  nrs.node_id,
  nrs.review_count,
  nrs.bayes_score,
  k.polarity,
  GROUP_CONCAT(k.keyword, ', ') AS keywords
FROM node_review_stats nrs
LEFT JOIN (
  SELECT *
  FROM node_review_keywords
  WHERE version = :version
  ORDER BY rank
) k
ON k.node_id = nrs.node_id
GROUP BY nrs.node_id, nrs.review_count, nrs.bayes_score, k.polarity
ORDER BY nrs.bayes_score DESC
LIMIT 10;
"""

preview = pd.read_sql_query(query, conn, params={"version": VERSION})

In [25]:
# ---- Verify join with node_review_stats ----
query = """
SELECT
  nrs.node_id,
  nrs.review_count,
  nrs.bayes_score,
  GROUP_CONCAT(CASE WHEN k.polarity = 'positive' THEN k.keyword END, ', ') AS pos_keywords,
  GROUP_CONCAT(CASE WHEN k.polarity = 'negative' THEN k.keyword END, ', ') AS neg_keywords
FROM node_review_stats nrs
LEFT JOIN node_review_keywords k
  ON k.node_id = nrs.node_id
 AND k.version = :version
GROUP BY nrs.node_id, nrs.review_count, nrs.bayes_score
ORDER BY nrs.bayes_score DESC

"""

preview = pd.read_sql_query(query, conn, params={"version": VERSION})

In [26]:
preview.sort_values('review_count', ascending=False)

,node_id,review_count,bayes_score,pos_keywords,neg_keywords
13,39,1075,0.014815,"간장, 교촌, 만족","간장, 교촌, 기대"
15,40,1068,0.005592,"교촌, 레드, 만족","교촌, 기대, 레드"
17,19,1019,-0.036133,"고추, 만족, 치킨","고추, 기대, 치킨"
11,27,1017,0.046967,"부추, 오징어, 풀무원","부추, 오징어, 풀무원"
16,105,994,-0.001001,"가쓰오, 우동, 정호영","가쓰오, 우동, 정호영"
22,26,196,-0.134328,"감자전, 만족, 풀무원","감자전, 기대, 풀무원"
21,2,183,-0.122340,"마라치, 양념치킨, 처갓집","마라치, 양념치킨, 처갓집"
18,109,182,-0.058824,"그릴, 만족, 비엔나","그릴, 기대, 비엔나"
8,111,180,0.064865,"만족, 이마트, 참치","기대, 이마트, 참치"
4,97,176,0.104972,"광어회, 마음, 이마트","광어회, 기대, 이마트"


In [27]:
#preview.loc[preview['node_id'] == 120]

In [28]:
results['keyword'].unique()

array(['마치', '기대', '실망', '만족', '마음', '마라치', '처갓집', '양념치킨', '소디', '교촌',
       '간장', '레드', '고추', '치킨', '풀무원', '감자전', '오징어', '부추', '바사', '불패',
       '자담', '피코', '커스터드', '쿠키', '산토리', '비루', '대가', '탕수육', '우촌', '주부',
       '비엔나', '양반', '그릴리', '부대', '이마트', '광어회', '정호영', '가쓰오', '우동', '그릴',
       '어세', '참치', '델리', '반반', '전골', '다시'], dtype=object)

In [31]:
def show_node_tables(results_df, source="count", top_n=TOP_N):
    view = results_df[results_df["source"] == source].copy()
    for (entity_id, node_name), group in view.groupby(["entity_id", "node_name"], dropna=False):
        title = f"Node {entity_id} — {node_name}" if pd.notna(node_name) else f"Node {entity_id}"
        print("=" * len(title))
        print(title)
        print("=" * len(title))
        for polarity, pgroup in group.groupby("polarity"):
            print(f"\n[{polarity}]")
            display(pgroup.sort_values("score", ascending=False).head(top_n)
                    [["keyword", "score", "source_count"]].reset_index(drop=True))
        print("\n")

show_node_tables(results, source="count", top_n=TOP_N)


Node 1 — 쏘마치 (BHC)

[negative]


,keyword,score,source_count
0,마치,81.0,94
1,기대,26.0,26
2,실망,26.0,26



[positive]


,keyword,score,source_count
0,마치,78.0,63
1,만족,33.0,33
2,마음,19.5,13




Node 2 — K마라치킨 (처갓집 양념치킨)

[negative]


...

,keyword,score,source_count
0,이마트,93.0,73
1,델리,93.0,73
2,만족,33.0,33




Node 120 — 이마트 회

[negative]


,keyword,score,source_count
0,이마트,90.0,92
1,기대,22.0,22
2,실망,22.0,22



[positive]


,keyword,score,source_count
0,이마트,78.5,66
1,만족,41.0,41
2,마음,16.5,11


## 5) Next steps (optional)

- Tune thresholds, confidence filter, or sentiment method/version.
- Try language-specific stopword lists or domain vocab.
- Use the keyword lists to select representative reviews per node/polarity.
